## Next Operator Action Prediction by Tim Tarver 

In [1]:
# Import required modules

import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import lightgbm as lgb

## Load the Datasets

In [3]:
# Load Datasets

# Load data
train = pd.read_csv("public/train.csv")
test = pd.read_csv("public/test.csv")
# Target
target = "Next_Operator_Action"
# Identity columns for submission
id_cols = ["Technician_ID", "Task_Session", "Event_Time_s"]
features = [c for c in train.columns if c not in id_cols + [target]]

## Separate Features and create new Feature

In [4]:
# Separate features

train = train.sort_values(
    ["Technician_ID","Task_Session","Event_Time_s"]
)

test = test.sort_values(
    ["Technician_ID","Task_Session","Event_Time_s"]
)

In [5]:
# Time since last event Feature creation

train["time_since_last_event"] = train.groupby(
    ["Technician_ID","Task_Session"]
)["Event_Time_s"].diff().fillna(0)

test["time_since_last_event"] = test.groupby(
    ["Technician_ID","Task_Session"]
)["Event_Time_s"].diff().fillna(0)

## Temporal Feature Engineering

In [6]:
# Temporal Feature Engineering

for df in [train, test]:

    df["time_delta"] = df.groupby(
        ["Technician_ID","Task_Session"]
    )["Event_Time_s"].diff().fillna(0)

    df["event_index"] = df.groupby(
        ["Technician_ID","Task_Session"]
    ).cumcount()

    df["session_progress"] = (
        df["event_index"] /
        df.groupby(["Technician_ID","Task_Session"])["event_index"].transform("max")
    )

## Technician Behavior Features

In [7]:
# Technician Behavior Features 

tech_stats = train.groupby("Technician_ID").agg(
    tech_event_count=("Event_Time_s","count"),
    tech_avg_time=("Event_Time_s","mean")
).reset_index()

train = train.merge(tech_stats,on="Technician_ID",how="left")
test = test.merge(tech_stats,on="Technician_ID",how="left")

## Detect Column Types

In [8]:
# Column Type Detection 

X = train[features]
y = train[target]

X_test = test[features]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()

print("Categorical:",cat_cols)
print("Numeric:",len(num_cols))

Categorical: ['Target_Component', 'Workshop_Lighting', 'Spoken_Command', 'Voice_Affect', 'Grasp_Type', 'Engagement_State', 'Operator_Affect', 'Task_Phase', 'Workflow_Type']
Numeric: 24


## Handle Missing Values 

In [9]:
# Handle Missing Values 

for col in num_cols:

    train[col+"_missing"] = train[col].isna().astype(int)
    test[col+"_missing"] = test[col].isna().astype(int)

    median = train[col].median()

    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)

for col in cat_cols:
    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

## Prepare Final Feature Matrices

In [10]:
# Final features

features = [c for c in train.columns if c not in id_cols + [target]]

X = train[features]
y = train[target]

X_test = test[features]

cat_features = [X.columns.get_loc(col) for col in cat_cols]

## Cross-Validation Training 

In [ ]:
# Cross-Validation Training 

kf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores = []

for train_idx,val_idx in kf.split(X,y):

    X_train,X_val = X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val = y.iloc[train_idx],y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=5000,
        depth=8,
        learning_rate=0.01,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val,y_val),
        verbose=False
    )

    preds = model.predict(X_val)

    score = accuracy_score(y_val,preds)

    scores.append(score)

print("Macro F1:",np.mean(scores))

## Train the Final Model

In [ ]:
# Train final model
model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=False
)

## Predict Test

In [ ]:
# Predict test
test_preds = model.predict(X_test)
test_preds = test_preds.flatten()

# Build the Submission

In [ ]:
# Save file
submission = test[id_cols].copy()

submission["Next_Operator_Action"] = test_preds

submission.to_csv("working/submission.csv", index=False)